# 00b · MLX Fundamentals: Lazy Evaluation, Unified Memory, `grad`, `compile`

**Time:** about 30 minutes · **Prerequisites:** basic NumPy

Notebook 00 claims MLX is lazy, uses unified memory, and transforms functions.
Here you see each claim demonstrated with measurements.

## Learning objectives
1. Explain **lazy evaluation**, and time MLX code correctly (`mx.eval`).
2. Use the **same array on CPU and GPU** without copying.
3. Compute gradients with `mx.grad` and check them numerically.
4. Speed up a function with `mx.compile`, and know when it helps.
5. Recognize the common pitfalls these features create.

In [ ]:
import time

import numpy as np
import mlx.core as mx
import mlx.nn as nn
import matplotlib.pyplot as plt

from mlx_nlp_utils import print_device_info, sanity_check

print_device_info()

## 1. Lazy evaluation

In NumPy, `c = a @ b` computes immediately. In MLX it records *what to
compute* and returns an array whose values do not exist yet. Work happens
when you ask for a value (`print`, `.item()`, `np.array(...)`) or call
`mx.eval(...)`.

🤔 **Predict:** how long will a 4096×4096 matrix multiply take to *return*,
compared with how long it takes to actually *compute*?

In [ ]:
a = mx.random.normal((4096, 4096))
b = mx.random.normal((4096, 4096))
mx.eval(a, b)                      # materialize the inputs first

start = time.perf_counter()
c = a @ b                          # builds a graph node; returns immediately
t_build = time.perf_counter() - start

start = time.perf_counter()
mx.eval(c)                         # now the work happens
t_compute = time.perf_counter() - start

print(f"'a @ b' returned in {t_build * 1000:8.3f} ms")
print(f"mx.eval(c) took     {t_compute * 1000:8.3f} ms")

The first number is graph-building only. **Benchmarking trap:** if you stop
the clock without `mx.eval`, you measure almost nothing:

In [ ]:
def bench(fn, repeats=10, warmup=2, evaluate=True):
    """Median milliseconds per call. With evaluate=False the work is NOT forced."""
    for _ in range(warmup):
        out = fn()
        if evaluate:
            mx.eval(out)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        out = fn()
        if evaluate:
            mx.eval(out)
        times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))

wrong = bench(lambda: a @ b, evaluate=False)
right = bench(lambda: a @ b, evaluate=True)
print(f"without mx.eval: {wrong:8.3f} ms   (wrong: measures graph building)")
print(f"with mx.eval:    {right:8.3f} ms   (right)")
sanity_check(right > 5 * wrong, "forgetting mx.eval makes a benchmark look far faster than it is")

### Why lazy?

Because MLX sees the whole graph before running, it can **fuse** operations
and skip work whose results are never used:

In [ ]:
x = mx.random.normal((2048, 2048))
mx.eval(x)

def unused_work():
    big = x @ x @ x @ x            # expensive...
    return x + 1                   # ...but never used: it is never computed

print(f"function that builds an unused 3-matmul chain: {bench(unused_work):.3f} ms "
      f"(vs. {bench(lambda: x @ x @ x @ x):.1f} ms to actually run that chain)")

**Consequence for training loops.** If you never evaluate, the graph grows
without bound: each step adds to it. This is why `train_model` calls
`mx.eval(loss, state)` after every update, which also bounds memory.

**Consequence for debugging.** An error in an operation can appear later,
at the point where something forces evaluation. When a traceback points at a
`print`, look at the lines that built the value.

## 2. Unified memory

On a discrete GPU, data must be copied between CPU RAM and GPU memory. On
Apple Silicon both processors address the same memory, so MLX arrays
are not tied to a device; you choose the device **per operation** with `stream`.

In [ ]:
print("default device:", mx.default_device())
big = mx.random.normal((2048, 2048))
mx.eval(big)

def matmul_on(device, n=3):
    stream = mx.gpu if device == "gpu" else mx.cpu
    return bench(lambda: mx.matmul(big, big, stream=stream), repeats=n)

cpu_ms, gpu_ms = matmul_on("cpu"), matmul_on("gpu")
print(f"2048×2048 matmul: CPU {cpu_ms:.1f} ms | GPU {gpu_ms:.1f} ms | GPU speedup {cpu_ms / gpu_ms:.1f}×")

# The SAME array is used on both devices with no transfer step:
r_cpu = mx.matmul(big, big, stream=mx.cpu)
r_gpu = mx.matmul(big, big, stream=mx.gpu)
print("max difference between CPU and GPU results:", mx.abs(r_cpu - r_gpu).max().item())

There is no `.to("cuda")`. The array `big` was read by both devices as is.
Small operations are often *faster* on the CPU because launching GPU work has a
fixed overhead: try the matrix size 64×64 below.

In [ ]:
sizes = [32, 128, 512, 1024, 2048]
cpu_times, gpu_times = [], []
for n in sizes:
    m = mx.random.normal((n, n)); mx.eval(m)
    cpu_times.append(bench(lambda: mx.matmul(m, m, stream=mx.cpu), repeats=20))
    gpu_times.append(bench(lambda: mx.matmul(m, m, stream=mx.gpu), repeats=20))

plt.figure(figsize=(7, 3.5))
plt.plot(sizes, cpu_times, 'o-', label='CPU'); plt.plot(sizes, gpu_times, 'o-', label='GPU')
plt.xscale('log', base=2); plt.yscale('log'); plt.xlabel('matrix size n (n×n @ n×n)'); plt.ylabel('ms')
plt.legend(); plt.title('Where does the GPU start to win?'); plt.show()

This is why notebook 04's tiny LSTM classifiers answer in ~2 ms regardless of
device, and why the large-matrix work in notebooks 07–08 benefits from the GPU.

## 3. Automatic differentiation: `mx.grad`

`mx.grad(f)` returns a *new function* that computes the gradient of `f`'s
output with respect to its first argument. No tape or `.backward()` call.

In [ ]:
def f(x):
    return mx.sum(x ** 2 + mx.sin(x))

grad_f = mx.grad(f)
x = mx.array([0.0, 1.0, 2.0])
print("analytic gradient 2x + cos(x):", np.array(2 * x + mx.cos(x)).round(4))
print("mx.grad(f)(x):               ", np.array(grad_f(x)).round(4))

### Check a gradient numerically

When you write a custom operation, compare its gradient with a finite
difference: $\frac{f(x+\epsilon) - f(x-\epsilon)}{2\epsilon}$.

In [ ]:
def numerical_grad(fn, x, eps=1e-3):
    out = []
    for i in range(x.size):
        step = mx.zeros_like(x).at[i].add(eps)
        out.append(((fn(x + step) - fn(x - step)) / (2 * eps)).item())
    return mx.array(out)

x = mx.random.normal((5,))
auto, numeric = grad_f(x), numerical_grad(f, x)
print("max |autodiff - finite difference| =", mx.abs(auto - numeric).max().item())
sanity_check(mx.abs(auto - numeric).max().item() < 1e-2, "autodiff matches the finite-difference gradient")

### Gradients of a model

`nn.value_and_grad(model, loss_fn)` does this for all of a module's trainable
parameters and returns a nested dictionary shaped like `model.parameters()`.
One gradient step on a toy linear regression:

In [ ]:
import mlx.optimizers as optim

mx.random.seed(0)
true_w = mx.array([[2.0], [-3.0]])
X = mx.random.normal((256, 2))
y = X @ true_w + 0.1 * mx.random.normal((256, 1))

model = nn.Linear(2, 1, bias=False)
loss_fn = lambda model, X, y: mx.mean((model(X) - y) ** 2)
loss_and_grad = nn.value_and_grad(model, loss_fn)
optimizer = optim.SGD(learning_rate=0.1)

losses = []
for _ in range(40):
    loss, grads = loss_and_grad(model, X, y)
    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state)       # without this the graph grows every step
    losses.append(loss.item())
print("learned weights:", np.array(model.weight).round(2).ravel(), "(true: [2, -3])")
print(f"loss {losses[0]:.3f} → {losses[-1]:.4f}")

## 4. `mx.compile`

`mx.compile` traces a function once and fuses its operations into fewer GPU
kernels. It pays off when a function chains many **small** operations, as
activation functions do: each separate operation otherwise reads and writes
the whole array through memory.

🤔 **Predict:** GELU is about eight element-wise operations. How much faster
will the compiled version be on a large array?

In [ ]:
def gelu(x):
    return x * 0.5 * (1 + mx.erf(x / mx.sqrt(mx.array(2.0))))

def many_small_ops(x):
    for _ in range(5):
        x = gelu(x) + 0.1 * mx.tanh(x)
    return x

compiled = mx.compile(many_small_ops)
data = mx.random.normal((4096, 4096)); mx.eval(data)
np.testing.assert_allclose(np.array(many_small_ops(data)), np.array(compiled(data)), atol=1e-4)

eager_ms = bench(lambda: many_small_ops(data), repeats=20)
compiled_ms = bench(lambda: compiled(data), repeats=20)
print(f"eager:    {eager_ms:7.2f} ms\ncompiled: {compiled_ms:7.2f} ms  ({eager_ms / compiled_ms:.1f}× faster)")

For a matrix multiply there is nothing to fuse: the work is one large
kernel, so compiling gains nothing.

In [ ]:
matmul_eager = bench(lambda: big @ big, repeats=10)
matmul_compiled = bench(lambda: mx.compile(lambda m: m @ m)(big), repeats=10)
print(f"matmul eager {matmul_eager:.2f} ms vs compiled {matmul_compiled:.2f} ms: about the same")

### Compiling a whole training step

Notebook 03's `train_model` compiles the entire update (forward, backward,
optimizer). A compiled function may only read and write MLX state that it is
told about, via `inputs=` and `outputs=`. Forget the optimizer state or
dropout's random state and the function silently misbehaves. That is exactly
what `make_train_step` captures:

In [ ]:
from mlx_nlp_utils import make_train_step

def make_run(compile_step, width, depth, batch):
    """Build a fresh MLP, a train step, and a batch; returns a function that runs n steps."""
    mx.random.seed(0)
    layers = [nn.Linear(256, width), nn.GELU()]
    for _ in range(depth):
        layers += [nn.Linear(width, width), nn.GELU(), nn.Dropout(0.1)]
    model = nn.Sequential(*layers, nn.Linear(width, 10))
    step, state = make_train_step(
        model, optim.Adam(learning_rate=1e-3),
        lambda m, X, y: nn.losses.cross_entropy(m(X), y, reduction='mean'), compile_step=compile_step)
    model.train()
    X, y = mx.random.normal((batch, 256)), mx.random.randint(0, 10, (batch,))
    mx.eval(state, X, y)
    def run(n):
        for _ in range(n):
            loss = step(X, y)
            mx.eval(loss, state)
        return loss.item()
    return run

# 1) Correctness: three steps from the same seed should give nearly the same loss.
print("loss after 3 steps: eager %.4f, compiled %.4f" % (make_run(False, 256, 2, 64)(3), make_run(True, 256, 2, 64)(3)))

# 2) Speed: best of 3 repeats of 30 steps, after a warm-up (compilation happens on the first call).
def ms_per_step(compile_step, width, depth, batch):
    run = make_run(compile_step, width, depth, batch)
    run(2)
    best = float('inf')
    for _ in range(3):
        start = time.perf_counter(); run(30)
        best = min(best, (time.perf_counter() - start) / 30 * 1000)
    return best

print(f"\n{'model (width×depth, batch)':<28} {'eager ms':>9} {'compiled ms':>12} {'speedup':>8}")
for width, depth, batch in [(64, 1, 32), (512, 2, 128), (2048, 6, 512)]:
    e, c = ms_per_step(False, width, depth, batch), ms_per_step(True, width, depth, batch)
    print(f"{f'{width}×{depth}, batch {batch}':<28} {e:>9.2f} {c:>12.2f} {e / c:>7.2f}×")

The speedup varies with model size and is **not guaranteed**: compiling
removes Python and kernel-launch overhead and fuses element-wise operations,
which helps most when a step consists of many small operations, and little when
a few large matrix multiplications dominate. Timings on small steps are also noisy.
The practical rule: **compile, then measure on your own model**; keep
`compile_step=False` available for debugging, since errors inside compiled code
are harder to read.

## 5. Pitfalls checklist

| Symptom | Cause | Fix |
|---|---|---|
| Benchmark is suspiciously fast | Lazy: work never forced | `mx.eval(result)` before stopping the clock |
| Memory grows every training step | Graph accumulates | `mx.eval(loss, model.parameters(), optimizer.state)` each step |
| Error appears at an unrelated line | Evaluation was deferred | Evaluate sooner while debugging (`mx.eval` after suspect ops) |
| Compiled function ignores dropout or optimizer updates | State not captured | Pass `inputs=`/`outputs=` (see `make_train_step`) |
| Compiled function slow on first call | Tracing/compilation cost | Warm up; avoid recompiling with changing shapes |
| Compiled step no faster (or slower) | Little to fuse, or timing noise | Measure several sizes and repeats; keep `compile_step=False` for debugging |
| Small ops slower on GPU | Launch overhead | Use `stream=mx.cpu`, or batch the work |

## 🧠 Exercises

**1. Fusion.** Add more element-wise operations to `many_small_ops` (loop 20
times instead of 5). 🤔 Predict: does the compile speedup grow?

<details><summary>What to look for</summary>

Usually yes: eager mode pays memory traffic per operation, while the compiled
version fuses the chain, so more fusable operations means a bigger gain.
</details>

**2. The crossover.** Using the CPU-vs-GPU plot, at what matrix size does the
GPU overtake the CPU on your machine? Does that match your expectation?

**3. Write a gradient.** Define `g(x) = mx.sum(mx.log(1 + mx.exp(x)))`
(softplus) and check `mx.grad(g)` against `mx.sigmoid(x)` and against the
finite-difference helper.

## ❓ FAQ

**Do I need `mx.eval` in normal code?** Not for correctness: printing, `.item()`,
and `np.array(...)` evaluate for you. You need it for benchmarking, for
bounding memory in loops, and when you want to control when work happens.

**Is MLX only for Apple Silicon?** The Metal GPU backend is. CPU-only builds
exist for other platforms, but this tutorial targets Apple Silicon.

**Next:** `01_Intent_Classification` puts these pieces to work on a real model.